## Problem 6: Applied Problem - Teenage Gambling

This problem is adapted from *Linear Models with R* by Julian J. Faraway.

The dataset `teengamb` (from R package `faraway`) concerns a study of teenage gambling in Britain.

### Tasks:

(a) Fit a regression model with gambling expenditure as the response and sex, socioeconomic status (on a percentile scale from 0-100), income (in pounds per week) and verbal score (correctly defined words out of 12) as predictors. Present the output of the summary command.

(b) What percentage of variation in the response is explained by these predictors?

(c) Which observation has the largest (positive) residual? Give the case number.

(d) Compute the mean and median of the residuals.

(e) Compute the correlation of the residuals with the fitted values.

(f) Compute the correlation of the residuals with the income.

(g) Fit a prediction interval for a new observation of a male teenager with a 60th percentile socioeconomic status, an income of 7 pounds per week, and a verbal score of 3 words out of 12.

(h) With all other predictors kept constant, what would be the difference in predicted gambling expenditure for a male compared to a female? Do you think it is reasonable to interpret this difference as a causal effect? Explain your reasoning.

In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr
from rpy2.robjects.conversion import localconverter

faraway = importr('faraway')
teengamb_r = ro.r['teengamb']

# Convert to pandas dataframe 
with localconverter(ro.default_converter + pandas2ri.converter):
    df = ro.conversion.rpy2py(teengamb_r)

print("Dataset Preview:")
print(df.head())
print("\n" + "="*80 + "\n")

print("Dataset Summary Statistics:")
print(df.describe())
print("\n" + "="*80 + "\n")

print("Dataset Info:")
print(df.info())
print("\n" + "="*80 + "\n")

print("Unique values in 'sex' variable:")
print(df['sex'].value_counts())



Dataset Preview:
   sex  status  income  verbal  gamble
1    1      51     2.0       8     0.0
2    1      28     2.5       8     0.0
3    1      37     2.0       6     0.0
4    1      28     7.0       4     7.3
5    1      65     2.0       8    19.6


Dataset Summary Statistics:
             sex     status     income     verbal      gamble
count  47.000000  47.000000  47.000000  47.000000   47.000000
mean    0.404255  45.234043   4.641915   6.659574   19.301064
std     0.496053  17.262944   3.551371   1.856558   31.515866
min     0.000000  18.000000   0.600000   1.000000    0.000000
25%     0.000000  28.000000   2.000000   6.000000    1.100000
50%     0.000000  43.000000   3.250000   7.000000    6.000000
75%     1.000000  61.500000   6.210000   8.000000   19.400000
max     1.000000  75.000000  15.000000  10.000000  156.000000


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
Index: 47 entries, 1 to 47
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------

In [23]:
# Part (a)
X = df[['sex', 'status', 'income', 'verbal']]
y = df['gamble']
X_with_const = sm.add_constant(X)

model = sm.OLS(y, X_with_const).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                 gamble   R-squared:                       0.527
Model:                            OLS   Adj. R-squared:                  0.482
Method:                 Least Squares   F-statistic:                     11.69
Date:                Mon, 16 Feb 2026   Prob (F-statistic):           1.81e-06
Time:                        14:37:49   Log-Likelihood:                -210.78
No. Observations:                  47   AIC:                             431.6
Df Residuals:                      42   BIC:                             440.8
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         22.5557     17.197      1.312      0.1

#### Part (b):

From the regression output in part (a):

- **R-squared = 0.527**
- **Adjusted R-squared = 0.482**

The four predictors (sex, socioeconomic status, income, and verbal score) explain 52.7% of the variation in teenage gambling expenditure. The remaining 47.3% of the variation is unexplained by the model and could be due to other factors not included (such as peer influence, family gambling habits, personality traits, etc.) or random variation. The adjusted R-squared of 48.2% accounts for the number of predictors in the model and penalizes for model complexity.

In [24]:
# Part (c)
residuals = model.resid
max_residual_idx = residuals.idxmax()
max_residual_value = residuals.max()

print(f"Case number with largest positive residual: {max_residual_idx}")
print(f"Residual value: {max_residual_value:.4f}")

Case number with largest positive residual: 24
Residual value: 94.2522


In [25]:
# Part (d)
residual_mean = residuals.mean()
residual_median = residuals.median()

print(f"Mean of residuals: {residual_mean:.10f}")
print(f"Median of residuals: {residual_median:.4f}")

Mean of residuals: 0.0000000000
Median of residuals: -1.4514


In [26]:
# Part (e)
fitted_values = model.fittedvalues
corr_resid_fitted = np.corrcoef(residuals, fitted_values)[0, 1]

print(f"Correlation between residuals and fitted values: {corr_resid_fitted:.10f}")

Correlation between residuals and fitted values: 0.0000000000


In [27]:
# Part (f)
corr_resid_income = np.corrcoef(residuals, df['income'])[0, 1]

print(f"Correlation between residuals and income: {corr_resid_income:.10f}")

Correlation between residuals and income: 0.0000000000


In [28]:
# Part (g)
new_obs = pd.DataFrame({
    'const': [1],
    'sex': [0],      
    'status': [60],   
    'income': [7],    
    'verbal': [3]     
})

prediction = model.get_prediction(new_obs)
pred_summary = prediction.summary_frame(alpha=0.05)

print(f"Predicted gambling expenditure: {pred_summary['mean'][0]:.2f} pounds per week")
print(f"\n95% Prediction Interval: [{pred_summary['obs_ci_lower'][0]:.2f}, {pred_summary['obs_ci_upper'][0]:.2f}]")

Predicted gambling expenditure: 51.55 pounds per week

95% Prediction Interval: [0.69, 102.41]


In [29]:
# Part (h)
sex_coefficient = model.params['sex']
sex_pvalue = model.pvalues['sex']

print(f"Sex coefficient: {sex_coefficient:.3f}")
print(f"p-value: {sex_pvalue:.3f}")

Sex coefficient: -22.118
p-value: 0.010


With all other predictors held constant, males spend 22.12 pounds per week more than females on gambling. This difference is statistically significant at the common level of 0.05. I don't believe that it is reasonable to interpret this as a causal effect because this is observational data with potential confounding variables that differ between males and females and also affect gambling behavior. To establish causation, we would need either a randomized experiment (impossible for sex) or much stronger quasi-experimental designs that carefully account for all potential confounders.